In [7]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, 
    confusion_matrix, 
    f1_score, 
    average_precision_score, 
    make_scorer
)

In [8]:
# Define the exact path to your processed data folder
PROCESSED_DATA_DIR = "Data/processed"

print("---> Loading processed data splits...")
X_train = pd.read_csv(os.path.join(PROCESSED_DATA_DIR, 'X_train_clean.csv'))
X_test = pd.read_csv(os.path.join(PROCESSED_DATA_DIR, 'X_test_clean.csv'))

# Convert y target dataframes into flat 1D arrays for Scikit-Learn
y_train = pd.read_csv(os.path.join(PROCESSED_DATA_DIR, 'y_train_clean.csv')).values.ravel()
y_test = pd.read_csv(os.path.join(PROCESSED_DATA_DIR, 'y_test_clean.csv')).values.ravel()

print(f"Data Loaded Successfully!")
print(f"Training shapes: X={X_train.shape}, y={y_train.shape}")
print(f"Testing shapes:  X={X_test.shape}, y={y_test.shape}")

---> Loading processed data splits...
Data Loaded Successfully!
Training shapes: X=(120889, 38), y=(120889,)
Testing shapes:  X=(30223, 38), y=(30223,)


In [9]:
def evaluate_model_cv(model, X_train, y_train):
    """
    Performs Stratified 5-Fold Cross-Validation on the training set
    and returns the mean and standard deviation of F1 and AUC-PR.
    """
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    scoring = {
        'f1': make_scorer(f1_score),
        'auc_pr': make_scorer(average_precision_score)
    }
    
    # n_jobs=-1 runs cross-validation folds in parallel using your Mac's CPU cores
    cv_results = cross_validate(model, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)
    
    return {
        'mean_f1': np.mean(cv_results['test_f1']),
        'std_f1': np.std(cv_results['test_f1']),
        'mean_auc_pr': np.mean(cv_results['test_auc_pr']),
        'std_auc_pr': np.std(cv_results['test_auc_pr'])
    }

In [10]:
# ---------------------------------------------------------------------
# Model 1: Baseline Logistic Regression
# ---------------------------------------------------------------------
print("\n---> Training Baseline: Logistic Regression...")
lr_model = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')

# Run Cross-Validation
lr_cv_metrics = evaluate_model_cv(lr_model, X_train, y_train)

# Fit on full training data and predict on holdout test set
lr_model.fit(X_train, y_train)
lr_preds = lr_model.predict(X_test)
lr_probs = lr_model.predict_proba(X_test)[:, 1]


# ---------------------------------------------------------------------
# Model 2: Ensemble Model (Random Forest)
# ---------------------------------------------------------------------
print("---> Training Ensemble: Random Forest (with basic tuning)...")
rf_model = RandomForestClassifier(
    n_estimators=150,        # Basic tuning: Number of decision trees
    max_depth=8,             # Basic tuning: Limits depth to prevent overfitting
    class_weight='balanced', # Penalizes misclassifying the minority (fraud) class
    random_state=42,
    n_jobs=-1
)

# Run Cross-Validation
rf_cv_metrics = evaluate_model_cv(rf_model, X_train, y_train)

# Fit on full training data and predict on holdout test set
rf_model.fit(X_train, y_train)
rf_preds = rf_model.predict(X_test)
rf_probs = rf_model.predict_proba(X_test)[:, 1]


---> Training Baseline: Logistic Regression...
---> Training Ensemble: Random Forest (with basic tuning)...


In [11]:
# ---------------------------------------------------------------------
# side-by-side performance breakdown
# ---------------------------------------------------------------------
# Calculate final holdout set scores
lr_f1_holdout = f1_score(y_test, lr_preds)
lr_auc_pr_holdout = average_precision_score(y_test, lr_probs)

rf_f1_holdout = f1_score(y_test, rf_preds)
rf_auc_pr_holdout = average_precision_score(y_test, rf_probs)

print("\n" + "="*60)
print(" FINAL MODEL PERFORMANCE COMPARISON REPORT ")
print("="*60)

results_table = pd.DataFrame({
    'Metric': ['CV Mean F1-Score', 'CV Mean AUC-PR', 'Holdout F1-Score', 'Holdout AUC-PR'],
    'Logistic Regression (Baseline)': [
        f"{lr_cv_metrics['mean_f1']:.4f} (±{lr_cv_metrics['std_f1']:.4f})",
        f"{lr_cv_metrics['mean_auc_pr']:.4f} (±{lr_cv_metrics['std_auc_pr']:.4f})",
        f"{lr_f1_holdout:.4f}",
        f"{lr_auc_pr_holdout:.4f}"
    ],
    'Random Forest (Ensemble)': [
        f"{rf_cv_metrics['mean_f1']:.4f} (±{rf_cv_metrics['std_f1']:.4f})",
        f"{rf_cv_metrics['mean_auc_pr']:.4f} (±{rf_cv_metrics['std_auc_pr']:.4f})",
        f"{rf_f1_holdout:.4f}",
        f"{rf_auc_pr_holdout:.4f}"
    ]
})
print(results_table.to_string(index=False))

print("\n### Confusion Matrices (Holdout Set) ###")
print(f"Logistic Regression Confusion Matrix:\n{confusion_matrix(y_test, lr_preds)}")
print(f"\nRandom Forest Confusion Matrix:\n{confusion_matrix(y_test, rf_preds)}")

print("\n### Detailed Classification Report (Random Forest Best Model) ###")
print(classification_report(y_test, rf_preds))


 FINAL MODEL PERFORMANCE COMPARISON REPORT 
          Metric Logistic Regression (Baseline) Random Forest (Ensemble)
CV Mean F1-Score               0.2733 (±0.0031)         0.6976 (±0.0095)
  CV Mean AUC-PR               0.1471 (±0.0020)         0.5785 (±0.0101)
Holdout F1-Score                         0.2730                   0.7014
  Holdout AUC-PR                         0.4281                   0.6353

### Confusion Matrices (Holdout Set) ###
Logistic Regression Confusion Matrix:
[[17684  9709]
 [  848  1982]]

Random Forest Confusion Matrix:
[[27392     1]
 [ 1301  1529]]

### Detailed Classification Report (Random Forest Best Model) ###
              precision    recall  f1-score   support

           0       0.95      1.00      0.98     27393
           1       1.00      0.54      0.70      2830

    accuracy                           0.96     30223
   macro avg       0.98      0.77      0.84     30223
weighted avg       0.96      0.96      0.95     30223

